**Photogrammetry** is the broad field encompassing techniques for extracting geometric measurements from photographs. **Structure from Motion (SfM)** is a specific approach within photogrammetry that recovers 3D scene structure *and* the camera motion from a set of 2D images. **Bundle Adjustment (BA)** is the non-linear optimization used within both photogrammetry and SfM to jointly refine the reconstructed 3D points and the camera parameters, minimizing the reprojection error to improve accuracy.


## Photogrammetry

   - **Definition**: Photogrammetry is the science of making measurements from photographs. It is broadly used in fields such as surveying, mapping, engineering, and archaeology.
   - **Application**: It involves taking measurements from photographs to create maps, 3D models, or drawings of real-world objects or scenes.
   - **Process**: It can use either single or multiple photographs and relies on the principles of perspective and projective geometry.
   - **Tools & Techniques**: It may involve various hardware and software tools, such as drones for aerial photography and specialized software for image processing and 3D reconstruction.

## Structure from Motion (SfM)

   - **Definition**: SfM is a photogrammetric method that reconstructs 3D structure from 2D image sequences. It sits at the intersection of computer vision and photogrammetry.
   - **Application**: Primarily used to reconstruct 3D models from photographs, particularly where traditional surveying methods are impractical.
   - **Process**: The pipeline detects and matches features across a series of images, estimates the camera poses (position and orientation), triangulates 3D points, and refines everything with bundle adjustment.
   - **Relation to Photogrammetry**: SfM is more automated and software-driven than traditional photogrammetry and is considered a modern advancement in the field.

## Bundle Adjustment
   - **Definition**: Bundle Adjustment is a non-linear optimization technique used in computer vision and photogrammetry.
   - **Application**: It refines a visual reconstruction to produce *jointly* optimal 3D structure and camera-parameter (pose, and optionally intrinsics and distortion) estimates.
   - **Process**: It adjusts the 3D point coordinates and the camera parameters by minimizing the reprojection error, i.e. the difference between the observed image points and the points obtained by projecting the estimated 3D structure back into each image.
   - **Relation to SfM and Photogrammetry**: Bundle Adjustment is the refinement step of SfM. In incremental SfM it is run repeatedly (after each new camera is added) as well as as a final global refinement; it is a critical part of both SfM and photogrammetry when accuracy matters.

### Reprojection error objective

Let camera $i$ have intrinsics $K_i$ and pose $(R_i, t_i)$, let $X_j$ be the 3D point $j$, and let $x_{ij}$ be the observed image location of point $j$ in image $i$. Bundle adjustment solves

$$\min_{\{K_i, R_i, t_i\},\ \{X_j\}} \ \sum_{i}\sum_{j} v_{ij}\,\bigl\lVert\, x_{ij} - \pi(K_i, R_i, t_i, X_j)\,\bigr\rVert^2,$$

where $\pi(\cdot)$ is the projection function that maps a 3D point into the image plane of camera $i$, and $v_{ij}\in\{0,1\}$ indicates whether point $j$ is actually observed in image $i$.

Key properties:

- **Joint optimization.** BA optimizes *all* camera poses (and, when self-calibrating, their intrinsics and distortion) together with *all* 3D point coordinates simultaneously. The name comes from the *bundles* of light rays leaving each 3D point and converging on each camera centre, which are "adjusted" to be mutually consistent.
- **Non-linear least squares.** Because $\pi$ is non-linear, BA is solved iteratively, typically with the **Levenberg-Marquardt** algorithm (a damped Gauss-Newton method). Widely used implementations include [Ceres Solver](http://ceres-solver.org/), g2o, and GTSAM.
- **Sparse Jacobian and the Schur complement.** Each residual depends on only one camera and one point, so the Jacobian $J$ - and hence the Gauss-Newton normal-equations matrix $J^\top J$ - has a characteristic sparse *arrowhead* block structure. Solvers exploit this with the **Schur complement**: the point (structure) block is block-diagonal and therefore cheap to invert, so it is eliminated first to form a much smaller *reduced camera system*, which is solved for the camera updates before back-substituting for the points. This is what makes BA tractable for problems with thousands of cameras and millions of points.
- **Robustification.** A robust loss (e.g. Huber or Cauchy) is usually wrapped around each residual to limit the influence of outlier correspondences.

Ref: [1](https://www.cs.cornell.edu/~snavely/bundler/bundler-v0.4-manual.html), [2](https://www.eecs.umich.edu/courses/eecs442-ahowens/fa21/slides/lec22-sfm.pdf), [3](https://ceres-solver.googlesource.com/ceres-solver/+/1.12.0/examples/snavely_reprojection_error.h)
,[4](http://ceres-solver.org/nnls_tutorial.html#bundle-adjustment), [5](https://homes.cs.washington.edu/~sagarwal/bal.pdf)

## Open-source SfM / MVS tools

- **[COLMAP](colmap.md)** - a general-purpose, end-to-end SfM *and* Multi-View Stereo (MVS) pipeline (feature extraction -> matching -> incremental mapping with bundle adjustment -> dense reconstruction). See the local [COLMAP notes](colmap.md) for a concrete, command-line SfM implementation.
- **CMVS-PMVS** - clustering and patch-based dense MVS. Refs: [1](https://github.com/pmoulon/CMVS-PMVS)
- **openMVG** - "open Multiple View Geometry", a library implementing the SfM pipeline. Refs: [1](https://opensourcephotogrammetry.blogspot.com/)


## Pipeline for SfM

**Feature Detection and Matching:**
First, features are detected in each image. Common algorithms include SIFT, SURF, and ORB.
These features are then matched between images to find correspondences. OpenCV's `cv::BFMatcher` or `cv::FlannBasedMatcher` can be used for this.

**Pose Estimation:**
With matched features, you estimate the relative pose between two images. `cv::recoverPose` is often used here: it computes the rotation and translation between two views given corresponding points and the essential matrix. The essential matrix can be estimated with `cv::findEssentialMat`.

```cpp
cv::Mat E = cv::findEssentialMat(points1, points2, focal, pp, cv::RANSAC, 0.999, 1.0, mask);
cv::recoverPose(E, points1, points2, R, t, focal, pp, mask);
```


**Incremental or Global Structure from Motion:**
- **Incremental SfM:** Start from an initial image pair, recover its relative pose and triangulate an initial set of 3D points, then add images one at a time. Each new camera is registered by 2D-3D correspondences (typically via PnP + RANSAC), new points are triangulated, and bundle adjustment is run to refine the structure and camera poses. It is robust and accurate but scales roughly super-linearly with the number of images. COLMAP is the canonical incremental-SfM system.
- **Global SfM:** All cameras are considered at once - first the relative rotations are averaged into a globally consistent set of orientations, then camera positions (translations) are solved together, and finally a single global bundle adjustment refines everything. It is faster and less prone to drift, but more sensitive to outlier matches.

## Noah Snavely reprojection error

The *Snavely reprojection error* is the specific projection/residual model introduced by Noah Snavely's **Bundler** and popularized by the **Bundle Adjustment in the Large (BAL)** dataset and the [Ceres Solver example](https://ceres-solver.googlesource.com/ceres-solver/+/1.12.0/examples/snavely_reprojection_error.h). Each camera is described by **9 parameters**: a 3-vector angle-axis (Rodrigues) rotation $\mathbf{r}$, a 3-vector translation $\mathbf{t}$, one focal length $f$, and two radial-distortion coefficients $k_1, k_2$.

Given a 3D point $X$, the predicted image coordinates are computed as:

1. **Rotate and translate** the point into the camera frame using the angle-axis rotation:
   $$P = R(\mathbf{r})\,X + \mathbf{t}.$$
2. **Perspective division.** Bundler's cameras look down the *negative* $z$ axis, so the point is projected with a sign flip:
   $$x' = -\frac{P_x}{P_z}, \qquad y' = -\frac{P_y}{P_z}.$$
3. **Radial distortion.** With $r^2 = x'^2 + y'^2$,
   $$\text{distortion} = 1 + k_1\,r^2 + k_2\,r^4.$$
4. **Apply focal length** to get the predicted pixel location:
   $$u = f \cdot \text{distortion} \cdot x', \qquad v = f \cdot \text{distortion} \cdot y'.$$

The residual passed to the solver is the difference between this prediction and the observed feature location, $(u - u_{\text{obs}},\ v - v_{\text{obs}})$. Because the model is analytic, its Jacobian is computed by automatic differentiation in Ceres, and the whole problem is solved as the sparse non-linear least-squares bundle adjustment described above.